# Track 06 — 오케스트레이션 & 멀티 에이전트

### 오케스트레이션이란?

오케스트레이션은 **하나의 LLM에 모든 일을 맡기지 않고, 작업을 역할(role)별로 나눈 뒤 단계 사이를 조율하는 방식**입니다. 한 모델이 계획·실행·검수를 모두 맡으면 책임 경계가 흐려지지만, **Planner(계획) → Executor(도구로 실행) → Critic(검수)** 로 나누면 단계마다 다른 프롬프트·권한·종료 조건을 줄 수 있고 단계별 trace를 남길 수 있습니다. 다만 역할을 나누면 LLM 호출이 늘어 지연·비용이 커지므로, **언제 단일 `ToolAgent`로 충분하고 언제 역할을 나눠야 하는지**가 이 트랙의 핵심 질문입니다.

### 이 노트북에서 보여줄 것

| Session | 무엇을 | 왜 |
|---|---|---|
| **1. Planner→Executor→Critic** | 사내 정책 메모 한 건을 세 역할로 나눠 실행하고 `workflow_trace.json`에 단계별 결과를 남김 | 역할 분리가 "계획·근거 수집·검수"를 어떻게 분담하는지, 구조화 출력(JSON 스키마 강제)이 어떻게 단계 간 계약으로 작동하는지 확인합니다. |
| **2. 라우터 핸드오프** | 질의를 5개 프로필로 — **두 가지 라우팅 타입**(결정적 규칙 / LLM 분류기) | 키워드가 분명하면 결정적 규칙으로 싸게, 모호하면 LLM 분류기로 의미 기반(심화) — **경쟁이 아니라 보완** 하는 두 방식을 봅니다. |
| **3. Single vs Multi** | 같은 작업을 단일 `ToolAgent`와 멀티 롤(executor+검증+보완)로 돌려 지연·품질·정답률을 비교하고 결정 가이드를 남김 | 잘 명세된 작업은 single이 빠르고 충분하지만, **검증 가능한 계산에서는 멀티 롤이 single의 미검증 오류를 결정적 검증기로 잡아낸다**는 점을 확인합니다. |

> 세 Session은 "역할을 나눈다 → 역할로 라우팅한다 → 나누는 것이 실제로 이득인지 따진다" 로 이어지는 하나의 흐름입니다.

### 이 노트북을 마치면

- Planner·Executor·Critic을 명시적으로 조율하고 단계별 trace를 남길 수 있습니다.
- `ThinkingRouter`의 거친 intent를 도메인 규칙과 결합해 하위 에이전트 프로필로 핸드오프할 수 있습니다.
- 지연 시간·성공 여부를 근거로 단일 vs 멀티를 선택하는 의사결정 트리를 만들 수 있습니다.

- **구성:** 각 `Session`은 가이드(텍스트) → 코드 → 출력 해석(텍스트) 순입니다.
- **전제:** LLM 키가 **필수**입니다(라이브 LLM 호출이 많고 Session 2는 라우터를 20회 호출). 키가 없으면 첫 셀의 `build_llm_from_env()` 가 커널을 종료합니다.
- **산출물:** `_out/workflow_trace.json`, `_out/routing_table.json`, `_out/comparison.json`, `_out/decision_tree.md`


In [ ]:
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import logging
import warnings

# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽도록 라이브러리 로그를 줄입니다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치가 필요합니다: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) Gate live LLM steps on API key presence.
# (kr) API 키 유무에 따라 라이브 LLM 단계를 제어합니다.
HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
ROOT = exaone.project_root()
TRACK06 = ROOT / "recipes" / "track06_orchestration_multi_agent"
DATA = TRACK06 / "data"
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)

# (en) Build the LLM client from root .env. NOTE: build_llm_from_env() calls sys.exit when key/base_url is missing,
#      so this notebook requires a key — HAS_API is recorded into traces, not used as a real runtime branch.
# (kr) 루트 .env로 LLM 클라이언트를 만듭니다. 주의: build_llm_from_env()는 키/base_url이 없으면 sys.exit 하므로 이 노트북에는 키가 필수입니다 —
#      HAS_API는 실제 분기가 아니라 trace 기록용입니다.
client = exaone.integrations.build_llm_from_env()

# (en) Shared JSON options for planner/critic structured calls.
# (kr) planner/critic 구조화 호출에서 공통으로 사용하는 JSON 옵션입니다.
json_opts = exaone.llm.ExaoneGenerateOptions(
    enable_thinking=False, max_new_tokens=512, response_format={"type": "json_object"}
)

print(
    "exaone",
    exaone.__version__,
    "| HAS_API =",
    HAS_API,
    "| model:",
    client.model if client else "(none)",
)

**출력 해석:** `HAS_API = True`와 `model: LGAI-EXAONE/K-EXAONE-236B-A23B`가 보이면 라이브 LLM 클라이언트와 데이터 경로가 준비된 것입니다.

- 이 줄이 출력되었다는 것은 키가 유효하다는 뜻입니다 — 키가 없으면 위 `build_llm_from_env()` 가 커널을 종료해 여기까지 오지 못합니다.
- `client.model`은 이후 모든 planner/executor/critic·라우터 호출이 공유하는 모델입니다.


### 참고: K-EXAONE 2.0 · agentic reasoning

`ToolAgent` 멀티턴 루프는 **agentic** 워크플로입니다. `enable_thinking=True`와 `preserve_thinking=True`를 **명시**하세요(환경 변수·`eval/exaone_api_kwargs.py`). payload에는 항상 실리며, **효과**는 K-EXAONE 2.0+ — 1.0은 무시합니다. chitchat·단발 QA는 둘 다 `False`(Track 01).

→ [`docs/k_exaone_2.md`](../../docs/k_exaone_2.md)


## Session 1. Planner → Executor → Critic

**테스트 시나리오** — `workflow_brief` 1건(재택·VPN 온보딩 메모) + `policy_snippets` 로 Planner→Executor→Critic 파이프라인을 확인합니다.

| 데이터 | 역할 |
|---|---|
| `workflow_brief.json` | 과제·제약·Critic 체크리스트 |
| `policy_snippets.json` | `lookup_policy` 근거 |

한 작업을 세 역할로 나눕니다.


### Session 1-1. 브리프 & 정책 스니펫 + Executor 도구

**하는 일:** 워크플로 브리프(과제·제약·Critic 체크리스트)와 정책 스니펫을 읽고, Executor가 사용할 `lookup_policy` 도구를 등록합니다.

**의미:** `lookup_policy`는 자유 텍스트 정규식이 아니라 **구조화된 필드(topic/id)** 로 스니펫을 고릅니다 — 도구 입력을 엄격한 스키마로 받는 방식(원칙 3)입니다.


In [ ]:
brief = json.loads((DATA / "workflow_brief.json").read_text(encoding="utf-8"))
SNIPPETS = json.loads((DATA / "policy_snippets.json").read_text(encoding="utf-8"))
print("시나리오:", brief["title"], "(workflow_brief.json)")
print("  목표:", brief["user_goal"][:70] + "…")
print("  Critic 체크:", len(brief.get("critic_checklist", [])), "항목")
print("  policy_snippets:", len(SNIPPETS), "건")
print("task:", brief["title"])


# (en) Match snippets by structured topic/id field, never regex on free text.
# (kr) 자유 텍스트 정규식이 아니라 구조화된 필드(topic/id)로 스니펫을 고릅니다.
def lookup_policy(args: dict) -> dict:
    topic = (args.get("topic") or "").strip()
    if not topic:
        return {"error": "topic is required"}
    hits = [
        s for s in SNIPPETS if topic in s.get("topic", "") or topic in s.get("id", "")
    ]
    return {"topic": topic, "hits": hits, "count": len(hits)}


LOOKUP_POLICY_SCHEMA = {
    "type": "function",
    "function": {
        "name": "lookup_policy",
        "description": "Look up HR/IT policy snippets by topic (재택, VPN, 연차, 보안).",
        "parameters": {
            "type": "object",
            "required": ["topic"],
            "properties": {"topic": {"type": "string"}},
            "additionalProperties": False,
        },
    },
}
registry = exaone.tools.ToolRegistry()
registry.register(
    exaone.tools.Tool(
        name="lookup_policy", schema=LOOKUP_POLICY_SCHEMA, execute=lookup_policy
    )
)
print("registered:", [s["function"]["name"] for s in registry.get_schemas()])

**출력 해석:** 브리프와 도구가 준비되었음을 보여줍니다.

- `시나리오: 2026 Q2 … 정책 요약 메모` · `Critic 체크: 4 항목` · `policy_snippets: 4 건` — 이후 세 역할이 다룰 입력의 규모입니다.
- `registered: ['lookup_policy']` — Executor가 호출할 수 있는 도구가 1개 등록되었습니다. 이 도구 하나만으로는 메모 작성과 검수 전체를 끝낼 수 없으므로, Session 1-3·1-4에서 사전 점검과 실제 실행을 이어갑니다.


### Session 1-2. Planner — 구조화된 계획

**하는 일:** Planner가 메모 작성 계획을 `research_questions / executor_brief / stop_when` 3개 필드의 JSON으로 생성합니다.

**의미:** `PLAN_SCHEMA`로 출력 형태를 강제하고 `StructuredOutputPipeline`이 검증·복구합니다. **단, 파이프라인은 사후 검증기일 뿐 모델을 유도하지 않으므로**, 프롬프트에 목표 JSON 형태를 명시해야 모델이 해당 키를 포함해 응답합니다.


In [ ]:
PLAN_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["research_questions", "executor_brief", "stop_when"],
    "properties": {
        "research_questions": {"type": "array", "items": {"type": "string"}},
        "executor_brief": {"type": "string"},
        "stop_when": {"type": "string"},
    },
}
planner_pipeline = exaone.output.StructuredOutputPipeline(
    json_schema=PLAN_SCHEMA, max_repair_attempts=1
)

plan = {}
plan_pipe = None
plan_latency_ms = 0.0
# (en) Declare the exact JSON shape in the prompt — StructuredOutputPipeline validates/repairs but does not steer the model.
# (kr) 정확한 JSON 형태를 프롬프트에 명시합니다 — StructuredOutputPipeline은 검증·복구만 수행하며 모델을 유도하지 않습니다.
planner_prompt = (
    "Plan a Korean internal policy memo.\n"
    f"Title: {brief['title']}\nGoal: {brief['user_goal']}\n"
    "Return ONLY JSON with EXACTLY these keys (no others): "
    '{"research_questions": [up to 3 short strings], "executor_brief": string, "stop_when": string}.'
)
t0 = time.monotonic()
plan_resp = client.chat(
    [
        exaone.llm.ExaoneMessage(role="system", content="Return JSON only."),
        exaone.llm.ExaoneMessage(role="user", content=planner_prompt),
    ],
    options=json_opts,
)
plan_pipe = planner_pipeline.process(plan_resp.content or "")
plan_latency_ms = (time.monotonic() - t0) * 1000
plan = plan_pipe.data if plan_pipe.success else {}
print("planner ok:", plan_pipe.success, "| ms:", round(plan_latency_ms, 1))
print("research_questions:", plan.get("research_questions"))

**출력 해석:** Planner가 유효한 계획 JSON을 생성했습니다.

- `planner ok: True` — `StructuredOutputPipeline`이 스키마 검증을 통과했습니다(프롬프트에 키를 명시했기 때문). `False`이면 `plan`이 비어 이후 Executor 브리프가 빈 상태로 넘어갑니다.
- `research_questions:` — 메모를 쓰기 전에 확인할 질문 목록(재택·VPN·보안 등)입니다. Executor가 이를 도구 조회의 길잡이로 씁니다.


### Session 1-3. 사전 점검 — `NextStepPlanner.screen_catalog`

**하는 일:** Executor를 만들기 전에, `screen_catalog`로 **현재 등록된 도구 카탈로그만으로 이 과제를 끝낼 수 있는지**를 미리 확인합니다.

**의미:** `answerable`은 "등록된 도구로 사용자 요청을 끝까지 처리할 수 있는가"를 따지는 보수적 판정입니다. 여기서는 설명이 빈약한 `lookup_policy` 하나만 카탈로그로 줘서 **메모 작성과 검수 전체**는 도구만으로 완결하기 어렵다고 판단해 `answerable: False`가 나오기 쉽습니다 — 오케스트레이터가 "도구 하나에 의존하지 말고 Planner·Executor·Critic으로 나누자"고 판단하게 하는 신호입니다. (이 셀은 사전 확인만 하며, 다음 셀 Executor는 이 도구로 실제 근거를 모읍니다.)


In [ ]:
catalog = [
    {
        "qualified_name": "tool.lookup_policy",
        "description": "lookup internal policy snippets",
    }
]
nsp = exaone.agents.NextStepPlanner(client, client.model)
screen = nsp.screen_catalog(query=brief["user_goal"], catalog=catalog)
print("answerable:", screen.answerable, "| suggested:", screen.suggested_tools)

**출력 해석:** 카탈로그 사전 점검 결과입니다.

- `answerable: False | suggested: []` — `lookup_policy` 도구 하나만으로는 "정책을 조회해 메모를 쓰고 검수까지"라는 요청을 끝까지 처리하기 어렵다는 보수적 판정입니다. 도구 조회는 **근거 수집**일 뿐이므로, 누락 없는 메모를 위해 Executor·Critic 단계가 이어집니다.
- 만약 `True`가 나오면(모델 비결정성) 도구만으로 충분하다고 본 것이며, 그래도 이후 단계가 초안 작성과 검수를 더해 품질을 높입니다.


### Session 1-4. Executor — 도구로 근거 모아 초안

**하는 일:** `ToolAgent`가 `lookup_policy`를 주제별로 호출해 정책 근거를 모은 뒤 한국어 초안을 작성합니다.

**의미:** Planner의 `executor_brief`가 입력으로 들어가고, Executor는 **도구 호출 → 근거 인용 → 초안 작성**을 한 번의 `run()` 호출로 수행합니다. Planner(계획)와 Executor(실행)의 권한이 분리되어 있음을 확인합니다.


In [ ]:
draft = ""
exec_result = None
exec_latency_ms = 0.0
executor_query = (
    f"{brief['user_goal']}\nPlanner: {plan.get('executor_brief', '')}\n"
    "각 토픽(재택, VPN, 연차, 보안)을 lookup_policy로 먼저 조회한 뒤, 조회 결과만으로 글머리표 5개 이내 메모를 쓰세요."
)
# (en) Plain tool loop (router off, planner-screen off): the executor calls the tool to gather evidence, then drafts.
#      Session 1-3 already demonstrated screen_catalog standalone, so we don't re-screen inside the executor here.
# (kr) 단순 도구 루프(라우터·planner 사전 점검 비활성화): executor는 도구로 근거를 모은 뒤 초안을 씁니다.
#      Session 1-3에서 screen_catalog를 따로 보였으므로 여기서는 executor 내부에서 다시 사전 점검하지 않습니다.
agent = exaone.agents.ToolAgent(
    tool_registry=registry,
    system_prompt=(
        "You do NOT know the internal policies. You MUST call lookup_policy for EACH topic "
        "(재택, VPN, 연차, 보안) and base every bullet only on the returned text. Cite the source id. Korean."
    ),
    use_thinking_router=False,
    use_next_step_planner=False,
    max_turns=8,
)
t0 = time.monotonic()
exec_result = agent.run(exaone.agents.AgentContext(query=executor_query), llm=client)
exec_latency_ms = (time.monotonic() - t0) * 1000
# (en) The ToolAgent returns a JSON object with an `answer` field; pass that raw content to the Critic.
# (kr) ToolAgent는 `answer` 필드를 가진 JSON을 반환합니다 — 그 원문을 Critic에 넘깁니다.
draft = exec_result.content or ""
print("executor ok:", exec_result.success, "| chars:", len(draft))

**출력 해석:** Executor가 근거를 모아 초안을 작성했습니다.

- `executor ok: True` — `ToolAgent.run`이 도구를 호출하고 한국어 초안(JSON `answer`)을 반환했습니다. `chars`는 초안 분량의 대략치로, 모델·실행마다 수백~천여 자로 달라집니다.
- 이 초안이 다음 Critic 단계의 검수 대상이 됩니다 — 즉 Executor 출력은 Critic 입력으로 전달되는 계약입니다.


### Session 1-5. Critic — 체크리스트로 검수

**하는 일:** Critic이 브리프의 체크리스트(재택 상한·VPN 2FA·연차 경로·보안 보고)로 초안을 항목별 검수해 `passed / items / summary` JSON을 생성합니다.

**의미:** Planner·Executor와 **다른 역할**(생성이 아니라 판정)입니다. 여기서도 프롬프트에 목표 JSON 형태를 명시해야 모델이 체크리스트 키 대신 `passed/items/summary` 구조로 응답합니다.


In [ ]:
CRITIC_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["passed", "items", "summary"],
    "properties": {
        "passed": {"type": "boolean"},
        "summary": {"type": "string"},
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["check", "ok", "note"],
                "properties": {
                    "check": {"type": "string"},
                    "ok": {"type": "boolean"},
                    "note": {"type": "string"},
                },
            },
        },
    },
}
critic_pipeline = exaone.output.StructuredOutputPipeline(
    json_schema=CRITIC_SCHEMA, max_repair_attempts=1
)

review = {}
critic_pipe = None
critic_latency_ms = 0.0
# (en) Declare the exact JSON shape; otherwise the model invents checklist-keyed JSON and strict validation rejects it.
# (kr) 정확한 JSON 형태를 명시합니다 — 그렇지 않으면 모델이 체크리스트 항목을 키로 한 JSON을 만들어 strict 검증에서 거부됩니다.
critic_prompt = (
    f"Checklist (one items entry each): {brief['critic_checklist']}\nDraft:\n{draft}\n"
    "Return ONLY JSON with EXACTLY these keys (no others): "
    '{"passed": boolean, "summary": string, "items": [{"check": string, "ok": boolean, "note": string}]}.'
)
t0 = time.monotonic()
critic_resp = client.chat(
    [
        exaone.llm.ExaoneMessage(role="system", content="Return JSON only."),
        exaone.llm.ExaoneMessage(role="user", content=critic_prompt),
    ],
    options=json_opts,
)
critic_pipe = critic_pipeline.process(critic_resp.content or "")
critic_latency_ms = (time.monotonic() - t0) * 1000
review = critic_pipe.data if critic_pipe.success else {}

# (en) Show WHAT the critic reviewed (the draft) and HOW it judged each checklist item.
# (kr) Critic이 무엇을(초안) 검수했고 각 체크리스트 항목을 어떻게 판정했는지 보여줍니다.
print("검수 대상 초안(앞부분):", " ".join((draft or "(없음)").split())[:110], "…")
print(
    "critic passed:",
    review.get("passed"),
    "| summary:",
    (review.get("summary") or "")[:80],
)
print("체크리스트 항목별 판정:")
for item in review.get("items", []):
    mark = "✓" if item.get("ok") else "✗"
    print(f"  [{mark}] {item.get('check', '')} → {item.get('note', '')}")

**출력 해석:** Critic이 **초안의 체크리스트 항목을 하나씩 판정한** 결과를 그대로 보여줍니다.

- `검수 대상 초안(앞부분): …` — Critic의 입력입니다(Executor가 만든 메모). 이 초안을 놓고 4개 항목을 검사합니다.
- `[✓] 재택 일수 상한이 명시되었는가 → 주 2일로 명시됨` 처럼 **항목마다 ✓/✗ 와 근거(note)**가 찍힙니다 — Critic이 무엇을 근거로 통과/실패로 봤는지가 드러납니다. 하나라도 ✗이면 `passed`가 보통 False가 됩니다(통과 여부는 모델 판단).
- 이것이 Executor(생성)와 분리된 **독립 검수**의 핵심입니다: 같은 초안을 다른 역할이 체크리스트로 재검토합니다. 프롬프트에 `passed/items/summary` 형태를 명시하지 않으면 모델이 체크리스트 항목을 키로 한 JSON을 만들어 strict 스키마에서 거부되므로, 이 명시가 검수 단계를 안정적으로 동작하게 하는 핵심입니다.


### Session 1-6. 산출물 — `workflow_trace.json`

**하는 일:** planner·executor·critic 세 단계의 success·latency·출력(그리고 executor의 `stop_reason`)을 `workflow_trace.json`에 저장합니다.

**의미:** 단계별 trace는 관측(Track 07)·회귀 테스트의 입력으로 쓰입니다. 어느 단계가 느리고 어느 단계에서 실패했는지를 한 파일로 추적합니다.


In [ ]:
trace = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "task_id": brief["task_id"],
    "has_api": HAS_API,
    "phases": {
        "planner": {
            "success": bool(plan_pipe.success) if plan_pipe else None,
            "latency_ms": round(plan_latency_ms, 1),
            "output": plan,
        },
        # (en) Only the executor (ToolAgent) carries an enrich stop_reason; planner/critic are raw chat calls.
        # (kr) executor(ToolAgent)만 enrich stop_reason을 가집니다 — planner/critic은 단순 chat 호출입니다.
        "executor": {
            "success": bool(exec_result.success) if exec_result else None,
            "latency_ms": round(exec_latency_ms, 1),
            "stop_reason": (
                exec_result.metadata.get("enrich_stop_reason") if exec_result else None
            ),
            "draft": draft[:2000],
        },
        "critic": {
            "success": bool(critic_pipe.success) if critic_pipe else None,
            "latency_ms": round(critic_latency_ms, 1),
            "review": review,
        },
    },
}
(out_dir / "workflow_trace.json").write_text(
    json.dumps(trace, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved:", (out_dir / "workflow_trace.json").resolve())

**출력 해석:** Session 1의 trace가 저장되었습니다.

- `saved: …/workflow_trace.json` — `phases.planner/executor/critic`의 success·latency와 executor의 `stop_reason`이 한 파일에 정리되었습니다.
- 세 단계 latency를 비교하면 보통 도구를 호출하는 **executor가 가장 오래** 걸립니다 — 역할 분리의 비용이 어디에 쏠리는지 확인할 수 있습니다.


### Session 1-7. 이 실행을 흐름도로 — 검사·반복까지

**하는 일:** 방금 처리한 메모가 단계를 어떻게 지났는지를, **검사(◇)와 반복(루프)**까지 포함해 Mermaid 흐름도로 그립니다. 이번 실행의 실제 데이터를 사용합니다.

**의미:** 파이프라인은 **일방향으로만 흐르지 않습니다.** ① `screen_catalog`는 "도구로 답할 수 있는가?"를 **검사**하고, ② Executor(`ToolAgent`)는 `도구 호출 → 결과 관찰 → 더 필요한지 판단`을 **반복**하다 충분해지면 초안을 씁니다(이번 실행에서는 그 루프가 도구를 여러 번 호출). ③ Critic은 체크리스트로 **검사**하여 통과 여부를 냅니다. 다만 Session 1의 Critic은 결과만 **기록**하고, "미통과 → 보완 후 재시도" 루프는 **Session 3의 multi 패턴**(executor → verify → revise)에서 확인합니다. (VS Code·JupyterLab 4에서는 다이어그램이 렌더링되고, 그 외 환경에서는 Mermaid 소스가 표시됩니다.)


In [ ]:
# (en) Render THIS Session 1 run as a Mermaid flow — including the checks (◇) and the Executor's tool loop.
# (kr) 이번 Session 1 실행을 Mermaid 흐름도로 표현합니다 — 검사(◇)와 Executor의 도구 반복 루프까지 담습니다.
from IPython.display import Markdown, display


def _esc(text):
    # (en) Keep node labels Mermaid-safe: drop quotes/newlines/brackets.
    # (kr) 노드 라벨이 Mermaid에서 안전하게 표시되도록: 따옴표·줄바꿈·대괄호 제거.
    return (
        str(text)
        .replace('"', "'")
        .replace("\n", " ")
        .replace("[", "(")
        .replace("]", ")")
        .strip()
    )


_n = len(plan.get("research_questions") or [])
_tools = exec_result.metadata.get("tool_invocations", 0) if exec_result else 0
_stop = exec_result.metadata.get("enrich_stop_reason") if exec_result else None
_items = review.get("items") or []
_n_ok = sum(1 for _it in _items if _it.get("ok"))
_passed = review.get("passed")

# (en) Decisions use {rhombus}; the Executor's reason->tool->observe cycle is a real loop (E -> O -> E).
# (kr) 판단 노드는 {마름모}; Executor의 reason->도구->관찰 사이클이 실제 루프입니다(E -> O -> E).
_flow = f"""flowchart TD
    B["📋 Brief: {_esc(brief['title'])}"] --> P["🧭 Planner<br/>research_questions {_n}개"]
    P --> S{{"🔎 screen_catalog<br/>answerable = {screen.answerable}?"}}
    S --> E["⚙️ Executor (ToolAgent 루프)<br/>이번 실행: 도구 {_tools}회 호출"]
    E -->|"lookup_policy"| O["👀 결과 관찰<br/>재택→주2일 · VPN→2FA …"]
    O --> E
    E -->|"충분해지면 종료<br/>stop = {_stop}"| D["📝 초안 {len(draft)}자"]
    D --> C{{"🧐 Critic 검수<br/>{_n_ok}/{len(_items)} 통과 · passed={_passed}"}}
    C -->|"✗ 미충족 항목"| R["⚠️ Session 1은 결과만 trace 기록<br/>(보완·재시도 루프는 → Session 3 multi)"]
    C -->|"✓ 전부 통과"| T["💾 workflow_trace.json"]
    R --> T"""

# (en) display() renders the mermaid block as a diagram in VS Code / JupyterLab 4 (else shows the source).
# (kr) display()는 VS Code·JupyterLab 4에서 Mermaid 블록을 다이어그램으로 렌더링합니다(그렇지 않으면 소스를 표시합니다).
display(Markdown(f"```mermaid\n{_flow}\n```"))

## Session 2. Router & 프로필 핸드오프 — 두 가지 라우팅 타입

**테스트 시나리오** — `routing_inputs.jsonl`(키워드가 분명한 질의 + **키워드 없는 모호한 질의** 를 섞음)을 5개 프로필로 보냅니다.

| 프로필 | 질문 유형 (예) |
|---|---|
| `faq` | HR·재택·VPN·보안 정책, 온보딩 |
| `data_analyst` | CSV·피벗·SQL, 수치 비교 |
| `code_review` | diff·race condition·리팩터·안전성 |
| `structured` | JSON·구조화 출력 |
| `general` | 잡담·요약·일반 지식 |

라우팅 방식 **두 가지** 를 단계적으로 봅니다:
- **Stage 1 — 결정적 규칙 라우터:** 코드/도메인 키워드·`has_tools` 로 라우팅. **싸고 빠르며, 키워드가 분명한 질의에 강합니다.** 키워드가 없으면 규칙의 영역 밖이라 미정(`None`)으로 남깁니다.
- **Stage 2 (심화) — LLM 분류기:** 키워드가 아니라 **뜻** 으로 5-way 분류하므로 모호한 질의까지 다룹니다 — 규칙보다 비싸지만 더 넓은 입력을 맡습니다.

둘은 **경쟁이 아니라 보완** 입니다: 키워드가 분명하면 규칙으로 싸게, 모호하면 LLM 으로 정확히. 입력 성격(목적)에 따라 어디까지 규칙으로 갈지가 달라집니다.


### Session 2-1. 프로필 + Stage 1 규칙 라우터

**하는 일:** 5개 프로필과 **결정적 규칙** 라우터 `route_rules` 를 정의합니다 — 규칙이 못 정하면 `None`(미정)을 돌려줍니다.

**의미:** 규칙은 코드/도메인 키워드와 `has_tools` 같은 **싸고 결정적인 신호** 만 봅니다. 키워드가 분명한 질의는 이걸로 충분하지만, 키워드 없는 질의는 `None`(미정)이 됩니다 — 그 미정 구간이 Stage 2 의 LLM 분류기가 맡을 자리입니다.


In [ ]:
PROFILE_SYSTEM = {
    "faq": "Internal FAQ — concise Korean.",
    "data_analyst": "Numbers/SQL/CSV — prefer tools.",
    "code_review": "Diffs, risks, tests.",
    "structured": "JSON-only answers.",
    "general": "Default assistant.",
}

DOMAIN_KW = ("연차", "재택", "VPN", "포털", "정책", "보안")
CODE_HINTS = (
    "diff",
    "pr ",
    "race condition",
    "리팩터",
    "refactor",
    "커버리지",
    "async",
    "blocking",
)


# (en) Stage 1 router: cheap deterministic signals only; returns None when nothing fires (undecided).
# (kr) Stage 1 라우터: 싸고 결정적인 신호만; 아무것도 안 걸리면 None(미정)을 돌려준다.
def route_rules(query, *, has_tools, intent):
    q_lower = query.lower()
    if any(k in q_lower for k in CODE_HINTS):
        return "code_review"
    if has_tools:
        return "data_analyst"
    if any(k in query for k in DOMAIN_KW):
        return "faq"
    if (intent or "").lower() == exaone.agents.SemanticIntent.STRUCTURED.value:
        return "structured"
    return None


router = (
    exaone.agents.ThinkingRouter(client=client, model=client.model) if HAS_API else None
)
print("profiles:", list(PROFILE_SYSTEM))

**출력 해석:** 5개 프로필과 라우팅 규칙이 준비되었습니다.

- `profiles: ['faq', 'data_analyst', 'code_review', 'structured', 'general']` — 핸드오프 후보입니다.
- 핵심은 `choose_profile`의 **판정 순서**입니다: `diff/PR/race condition` 표식 → `has_tools` → 도메인 키워드(재택·VPN·연차…) → `structured` intent → `analytical` intent → general. 결정적 신호를 거친 intent보다 먼저 적용해야 오분류가 줄어듭니다.


### Session 2-2. Stage 1 — 결정적 규칙 라우터

**하는 일:** 입력 전체를 `route_rules` 로 라우팅합니다. 규칙이 미정(`None`)으로 남긴 질의는 일단 `general` 로 두고 `expected_profile` 과 비교합니다.

**입력:** `data/routing_inputs.jsonl`

**의미:** 규칙은 **키워드가 분명한 질의에 강합니다.** 키워드가 없는 질의는 규칙의 영역 밖이라 미정으로 남고, naive 하게 `general` 로 두면 그중 general 이 아닌 것은 빗나갑니다 — 그 미정 구간을 Stage 2 가 의미로 맡습니다.


In [ ]:
rows = []
routing_lines = (DATA / "routing_inputs.jsonl").read_text(encoding="utf-8").splitlines()
print(
    "시나리오: 라우팅 입력",
    len([l for l in routing_lines if l.strip()]),
    "건 (routing_inputs.jsonl)",
)
for line in routing_lines:
    if not line.strip():
        continue
    item = json.loads(line)
    decision = router.route(
        item["query"],
        has_tools=item.get("has_tools", False),
        has_context=item.get("has_context", False),
    )
    rule_profile = route_rules(
        item["query"],
        has_tools=item.get("has_tools", False),
        intent=decision.semantic_intent,
    )
    stage1 = rule_profile or "general"  # (kr) 미정은 naive 하게 general 로
    rows.append(
        {
            "id": item["id"],
            "expected_profile": item.get("expected_profile"),
            "rule_profile": rule_profile,
            "stage1_profile": stage1,
            "semantic_intent": decision.semantic_intent,
            "enable_thinking": decision.enable_thinking,
            "stage1_match": stage1 == item.get("expected_profile"),
            "query": item["query"],
        }
    )
stage1_match = sum(1 for r in rows if r["stage1_match"])
undecided = [r for r in rows if r["rule_profile"] is None]
misrouted = [r for r in undecided if r["expected_profile"] != "general"]
print(
    f"Stage 1 (규칙만) match: {stage1_match}/{len(rows)} · 규칙 미정 {len(undecided)}건"
)
print(
    f"키워드가 없어 미정 → naive 하게 general 로 보냄 (그중 {len(misrouted)}건 오분류). 예:"
)
for r in misrouted[:3]:
    q = " ".join(r["query"].split())
    q = q[:30] + "…" if len(q) > 30 else q
    print(f"  {r['id']}  \"{q}\"  → general  (기대: {r['expected_profile']})")

**출력 해석:** 규칙 라우터는 키워드 질의에 강하고, 키워드 없는 질의는 미정으로 남깁니다.

- `Stage 1 (규칙만) match: X/N` — 코드/도메인 키워드·`has_tools` 가 있는 질의는 규칙이 싸고 정확하게 보냅니다.
- 출력의 `r21  "법인카드는 어떻게…"  → general  (기대: faq)` 처럼, 키워드 없는 질의는 규칙의 영역 밖이라 미정이 되고 naive 하게 general 로 새면 빗나갑니다(faq/data/code 모호 질의).
- 즉 규칙은 키워드가 분명한 만큼 강하고 싸지만, 입력이 모호하면 **뜻** 을 봐야 합니다 — 그 자리를 Stage 2 의 LLM 분류기가 맡습니다.


### Session 2-3. Stage 2 (심화) — LLM 분류기 (의미 기반)

**하는 일:** Stage 1 이 미정으로 남긴 질의에만 **LLM 5-way 분류기** `classify_profile` 을 호출합니다(규칙이 정한 건 그대로 둠).

**의미:** 키워드가 아니라 **뜻** 으로 분류하므로, 키워드 없는 모호한 질의까지 다룹니다. `ThinkingRouter` 의 거친 3분류 intent 로는 faq/data/code 를 못 가르지만(키워드 없으면 대개 analytical), 5-way 분류기는 가릅니다 — 규칙보다 비싸지만 더 넓은 입력을 맡는 **두 번째 라우팅 타입** 입니다.


In [ ]:
PROFILE_DESC = (
    "faq: internal HR/IT policy or onboarding Q&A; "
    "data_analyst: numbers, statistics, comparisons, SQL/tables; "
    "code_review: code quality, bugs, security, refactoring, tests; "
    "structured: the user demands the answer ONLY in a structured/JSON/table format; "
    "general: chitchat, brainstorming, general knowledge, summarization, anything else"
)
CLASSIFY_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["profile"],
    "properties": {"profile": {"type": "string", "enum": list(PROFILE_SYSTEM)}},
}
classify_pipeline = exaone.output.StructuredOutputPipeline(
    json_schema=CLASSIFY_SCHEMA, max_repair_attempts=1
)


# (en) Stage 2: an LLM 5-way profile classifier — called ONLY where the cheap rules were undecided.
# (kr) Stage 2: LLM 5-way 프로필 분류기 — 규칙이 미정인 곳에서만 호출한다.
def classify_profile(query):
    prompt = (
        f"Classify the query into exactly one profile.\nProfiles — {PROFILE_DESC}.\n"
        f'Query: {query}\nReturn ONLY JSON: {{"profile": one of the five}}.'
    )
    resp = client.chat(
        [
            exaone.llm.ExaoneMessage(role="system", content="Return JSON only."),
            exaone.llm.ExaoneMessage(role="user", content=prompt),
        ],
        options=json_opts,
    )
    out = classify_pipeline.process(resp.content or "")
    return out.data.get("profile") if out.success else "general"


for r in rows:
    # (en) keep the rule decision when it fired; otherwise let the LLM classifier decide.
    # (kr) 규칙이 정한 건 그대로, 미정만 LLM 분류기가 정한다.
    r["stage2_profile"] = (
        r["rule_profile"]
        if r["rule_profile"] is not None
        else classify_profile(r["query"])
    )
    r["stage2_match"] = r["stage2_profile"] == r["expected_profile"]
stage2_match = sum(1 for r in rows if r["stage2_match"])
recovered = [r for r in rows if not r["stage1_match"] and r["stage2_match"]]
print(
    f"Stage 2 (규칙 + LLM 분류기) match: {stage2_match}/{len(rows)}  (Stage 1 은 {stage1_match})"
)
print(f"LLM 분류기가 바로잡은 질의 {len(recovered)}건 — 같은 질의, Stage 1 → Stage 2:")
for r in recovered:
    q = " ".join(r["query"].split())
    q = q[:30] + "…" if len(q) > 30 else q
    print(f"  {r['id']}  \"{q}\"  {r['stage1_profile']} ✗ → {r['stage2_profile']} ✓")

**출력 해석:** LLM 분류기가 규칙의 영역 밖이던 모호한 질의를 의미로 분류합니다.

- `Stage 2 … match: Y/N (Stage 1 은 X)` — 규칙만일 때 X 였던 일치율이, 미정 질의에 LLM 분류기를 더하자 Y 로 올랐습니다.
- `r21  "법인카드는 어떻게…"  general ✗ → faq ✓` 처럼, **같은 질의가 Stage 1 에선 general 로 갔다가 Stage 2 에선 올바른 전문 프로필** 로 갑니다 — 출력에 질의 텍스트와 함께 바뀐 라우팅이 한 줄씩 찍힙니다.
- 핵심 교훈: 두 타입은 **보완 관계** 입니다 — **키워드가 분명하면 싸고 결정적인 규칙으로, 모호하면 LLM 분류기로.** 입력 성격에 맞춰 둘을 결합하면 비용과 정확도를 함께 잡습니다.


### Session 2-4. 산출물 — `routing_table.json`

**하는 일:** 두 단계 결과(Stage 1 규칙 / Stage 2 규칙+LLM)를 행별로 `routing_table.json` 에 저장합니다.

**의미:** 어떤 질의를 규칙이 정했고 어떤 질의를 LLM 분류기가 정했는지, 단계별 일치를 사후 추적합니다(관측·회귀 입력).


In [ ]:
report = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "has_api": HAS_API,
    "n_inputs": len(rows),
    "stage1_match_count": stage1_match,
    "stage2_match_count": stage2_match,
    "rows": [
        {
            k: r[k]
            for k in (
                "id",
                "expected_profile",
                "rule_profile",
                "stage1_profile",
                "stage2_profile",
                "semantic_intent",
                "stage1_match",
                "stage2_match",
            )
        }
        for r in rows
    ],
}
(out_dir / "routing_table.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved:", (out_dir / "routing_table.json").resolve())

**출력 해석:** 두 단계 라우팅 결과가 저장되었습니다.

- `saved: …/routing_table.json` — 각 행에 `rule_profile`(규칙)·`stage2_profile`(규칙+LLM)·단계별 일치가 있어, **규칙이 어디까지 하고 LLM 이 어디서 값을 했는지** 한 줄씩 추적됩니다.
- 이 표가 "규칙 + LLM 폴백" 2단 라우팅의 비용·정확도 근거가 됩니다.


## Session 3. Single vs Multi — 언제 무엇을 쓸까

**목표** — 같은 작업을 **단일 `ToolAgent`**와 **멀티 롤(executor → 검증 → 보완)**로 돌려 **장단점을 수치로** 비교하고, "언제 무엇을 쓸지" 결정 가이드를 만듭니다.

| | single | multi (executor+verify+revise) |
|---|---|---|
| 속도 | **빠름** (1패스) | 느림 (검증·보완 호출) |
| 잘 명세된 작업 | 충분 | 같은 품질, 지연만 ↑ |
| 검증 가능·오류 비용이 큰 작업 | 틀려도 그대로 내보냄 | **검증기가 오류를 잡아 보완** |

두 파트로 확인합니다 — **Part A**(잘 명세된 정책 작업: single 우위) · **Part B**(검증 가능한 계산: 멀티 롤의 검증 보장).


### Session 3-1. 도구·러너·검증기

**하는 일:** 두 도구(`lookup_policy`·`calc`)와, 비교할 두 러너를 정의합니다 — **single**(`ToolAgent` 1회, 초안을 그대로 제출)과 **multi**(executor → 독립 검증 → 문제가 있으면 필요한 부분만 보완).

**의미:** single·multi는 **같은 도구와 같은 시스템 프롬프트**를 사용합니다 — 차이는 오직 **검증 단계의 유무**입니다. 그래야 "멀티 롤이 더해 주는 가치"가 프롬프트 차이가 아니라 **조율(verify→revise)**에서 나온다는 점을 공정하게 확인합니다.


In [ ]:
import re

# (en) Two tools: policy lookup (well-specified tasks) and a calculator (verifiable arithmetic).
# (kr) 두 도구: 정책 조회(명세된 작업)와 계산기(검증 가능한 산술).
POLICY = json.loads((DATA / "policy_snippets.json").read_text(encoding="utf-8"))


def lookup_policy(args: dict) -> dict:
    topic = (args.get("topic") or "").strip()
    return {"topic": topic, "hits": [s for s in POLICY if topic in s.get("topic", "")]}


def calc(args: dict) -> dict:
    # (en) Normalize the commas/×/÷ a model may emit, then eval arithmetic only (no builtins, no ** DoS).
    # (kr) 모델이 낼 수 있는 콤마·×·÷를 정규화한 뒤 산술만 eval합니다(빌트인 없음, ** 금지).
    expr = (
        str(args.get("expression", ""))
        .replace(",", "")
        .replace("×", "*")
        .replace("÷", "/")
        .strip()
    )
    if (
        not expr
        or len(expr) > 100
        or "**" in expr
        or any(ch not in "0123456789+-*/(). " for ch in expr)
    ):
        return {"error": "invalid expression"}
    try:
        return {"result": eval(expr, {"__builtins__": {}})}
    except Exception as exc:
        return {"error": str(exc)}


def _registry(name, desc, params, fn):
    reg = exaone.tools.ToolRegistry()
    reg.register(
        exaone.tools.Tool(
            name=name,
            schema={
                "type": "function",
                "function": {"name": name, "description": desc, "parameters": params},
            },
            execute=fn,
        )
    )
    return reg


reg_policy = _registry(
    "lookup_policy",
    "Look up an HR/IT policy snippet by topic (재택, VPN, 연차, 보안).",
    {
        "type": "object",
        "required": ["topic"],
        "properties": {"topic": {"type": "string"}},
        "additionalProperties": False,
    },
    lookup_policy,
)
reg_calc = _registry(
    "calc",
    "Evaluate ONE arithmetic expression, e.g. 3*132000+5*27500.",
    {
        "type": "object",
        "required": ["expression"],
        "properties": {"expression": {"type": "string"}},
        "additionalProperties": False,
    },
    calc,
)

EXEC_SYS = "Use the available tools to ground facts and compute every number exactly. Be complete and precise. Write prose in Korean."


# (en) Extractors: memo task -> answer text; calc task -> {subtotal,vat,total_with_vat,average} dict.
# (kr) 추출기: 메모 작업 -> answer 텍스트; 계산 작업 -> {subtotal,vat,total_with_vat,average} 딕셔너리.
def extract_memo(res):
    data = res.structured
    if isinstance(data, dict) and isinstance(data.get("answer"), str):
        return data["answer"]
    return res.content or ""


def extract_calc(res):
    # (en) ToolAgent wraps the model's JSON under structured["answer"] or raw content; recover the first {...} object.
    # (kr) ToolAgent는 모델 JSON을 structured["answer"] 또는 원문에 담습니다; 첫 {...} 객체를 회수합니다.
    data = res.structured
    if isinstance(data, dict) and ("total_with_vat" in data or "subtotal" in data):
        return data
    candidates = []
    if isinstance(data, dict) and isinstance(data.get("answer"), str):
        candidates.append(data["answer"])
    candidates.append(res.content or "")
    for text in candidates:
        match = re.search(r"\{.*\}", text, re.S)
        if match:
            try:
                parsed = json.loads(match.group())
                if isinstance(parsed, dict):
                    return parsed
            except json.JSONDecodeError:
                continue
    return {}


def _agent(registry):
    return exaone.agents.ToolAgent(
        tool_registry=registry,
        system_prompt=EXEC_SYS,
        use_thinking_router=False,
        use_next_step_planner=False,
        max_turns=8,
        max_tool_invocations=12,
    )


# (en) single = one ToolAgent pass (ships its draft as-is — no second look).
# (kr) single = ToolAgent 1회(검토 없이 초안을 그대로 제출).
def run_single(query, registry, extract):
    t0 = time.monotonic()
    out = extract(
        _agent(registry).run(exaone.agents.AgentContext(query=query), llm=client)
    )
    return {"answer": out, "latency_ms": round((time.monotonic() - t0) * 1000, 1)}


# (en) multi = executor draft -> independent verify -> targeted revise only if the verifier flags an issue.
# (kr) multi = executor 초안 -> 독립 검증 -> 검증기가 문제를 지적할 때만 필요 부분 보완.
def run_multi(query, registry, extract, verify):
    t0 = time.monotonic()
    draft = extract(
        _agent(registry).run(exaone.agents.AgentContext(query=query), llm=client)
    )
    issues = verify(draft)
    if issues:
        revise_q = (
            f"{query}\n이전 답: {json.dumps(draft, ensure_ascii=False)}\n"
            f"검증에서 발견된 문제: {issues}\n도구로 다시 확인·보완해 올바른 답으로만 다시 답하세요."
        )
        draft = extract(
            _agent(registry).run(exaone.agents.AgentContext(query=revise_q), llm=client)
        )
    final_issues = verify(draft)
    return {
        "answer": draft,
        "latency_ms": round((time.monotonic() - t0) * 1000, 1),
        "issues_found": issues,
        "revised": bool(issues),
        "final_ok": not final_issues,
    }


print("runners ready | tools: lookup_policy, calc")

**출력 해석:** 비교 준비가 끝났습니다.

- `runners ready | tools: lookup_policy, calc` — 정책 조회·산술 두 도구와 `run_single`/`run_multi`가 준비됐습니다.
- 두 러너의 **유일한 차이는 검증 단계**입니다: `run_multi`는 executor의 초안을 독립 검증기로 확인하고, 문제가 있을 때만 한 번 더 도구로 보완합니다. `run_single`은 초안을 그대로 냅니다.


### Session 3-2. Part A — 잘 명세된 작업: single이 빠르고 충분

**하는 일:** 간단한 FAQ와 잘 명세된 정책 메모(4개 주제)를 single vs multi로 돌려 **지연 시간 + 품질(커버리지)** 을 비교합니다.

**의미:** 모델이 한 번에 잘 처리하는 작업에서는 multi의 검증이 **잡아낼 항목이 없어** 지연만 늘립니다. "복잡해 보이는 메모"도 잘 명세되면 single로 충분하다는 점을 수치로 확인합니다.


In [ ]:
# (en) Deterministic quality scorer: which required policy facts did the memo include?
# (kr) 결정적 품질 평가기: 메모에 필수 정책 사실이 포함되었는가?
FACTS = {
    "재택": ["2일", "이틀"],
    "VPN": ["2fa", "mfa"],
    "연차": ["포털"],
    "보안": ["security@", "핫라인", "hotline", "#sec"],
}


def missing_facts(memo):
    text = memo.lower()
    return [
        name for name, keys in FACTS.items() if not any(k.lower() in text for k in keys)
    ]


part_a = [
    {
        "id": "simple_faq",
        "max": 1,
        "query": "연차 신청은 어디서 하나요? lookup_policy로 확인 후 한 문장으로 답하세요.",
        "verify": lambda m: [] if "포털" in m else ["연차 신청 경로(HR 포털) 누락"],
    },
    {
        "id": "policy_memo",
        "max": 4,
        "query": brief["user_goal"]
        + " 재택·VPN·연차·보안 4가지를 모두 포함하고 각 항목 출처를 인용하세요.",
        "verify": lambda m: [f"{name} 미언급" for name in missing_facts(m)],
    },
]
print("시나리오: Part A 잘 명세된 작업", len(part_a), "건")
results_a = []
for task in part_a:
    s = run_single(task["query"], reg_policy, extract_memo)
    m = run_multi(task["query"], reg_policy, extract_memo, task["verify"])
    s_q = task["max"] - len(task["verify"](s["answer"]))
    m_q = task["max"] - len(task["verify"](m["answer"]))
    results_a.append(
        {
            "task_id": task["id"],
            "single": {
                "latency_ms": s["latency_ms"],
                "quality": f"{s_q}/{task['max']}",
            },
            "multi": {
                "latency_ms": m["latency_ms"],
                "quality": f"{m_q}/{task['max']}",
                "revised": m["revised"],
            },
        }
    )
    print(
        f"[{task['id']}] single {s['latency_ms']}ms q={s_q}/{task['max']} | multi {m['latency_ms']}ms q={m_q}/{task['max']} (검증 보완={m['revised']})"
    )

**출력 해석:** 잘 명세된 두 작업에서는 single이 빠르고 품질도 충분합니다.

- `[simple_faq]`·`[policy_memo]` 모두 single·multi의 **품질이 같습니다**(faq 1/1, memo 보통 4/4) — 모델이 한 번에 잘 처리하므로 multi의 검증기가 보완할 항목이 없습니다(`검증 보완=False`).
- 다만 multi는 검증 호출만큼 **더 느립니다**. 즉 이런 작업에서는 멀티 롤은 **순손실**(지연 ↑, 품질 동일) — **기본은 single**입니다.


### Session 3-3. Part B — 검증 가능한 계산: 멀티 롤은 출력을 검증한다

**하는 일:** 출장비 정산(곱셈·합산·부가세·평균)을 single vs multi로 돌리고, **결정적 검증기**(파이썬이 정답을 재계산)가 틀린 값을 어떻게 잡아내는지 확인합니다.

**입력:** `data/expense_claims.json` (정답은 파이썬으로 계산)

**의미:** single은 답을 **검증 없이 내보냅니다**. multi는 같은 답을 결정적 검증기로 확인하고, 틀리면 `calc`로 보완을 시도한 뒤 다시 검증합니다 — **모든 출력이 검증을 거친다**는 점이 멀티 롤의 값입니다(LLM은 다단계 산술을 종종 틀리므로).


> **실행 팁:** 기본은 **1 trial**(자동 실행·CI 친화). 3회 통계 비교가 필요하면 `.env`에 `ORCHESTRATION_CALC_TRIALS=3`을 설정하세요.

In [ ]:
# (en) Ground truth computed in Python — the deterministic verifier the LLM output is checked against.
# (kr) 파이썬으로 계산한 정답 — LLM 출력을 대조하는 결정적 검증기.
claims = json.loads((DATA / "expense_claims.json").read_text(encoding="utf-8"))
gt_person = {
    c["name"]: sum(i["qty"] * i["unit_price"] for i in c["items"])
    for c in claims["claims"]
}
GT_SUBTOTAL = sum(gt_person.values())
GT_VAT = round(GT_SUBTOTAL * claims["vat_rate"])
GT_TOTAL = GT_SUBTOTAL + GT_VAT
GT_AVG = round(GT_SUBTOTAL / len(gt_person))
GROUND_TRUTH = {
    "subtotal": GT_SUBTOTAL,
    "vat": GT_VAT,
    "total_with_vat": GT_TOTAL,
    "average": GT_AVG,
}


def verify_calc(ans):
    if not isinstance(ans, dict) or "total_with_vat" not in ans:
        return ["계산 결과 파싱 실패"]
    return [
        f"{k} {ans.get(k)} != 검증값 {v}"
        for k, v in GROUND_TRUTH.items()
        if ans.get(k) != v
    ]


def _fmt(c):
    return ", ".join(
        f"{i['label']} {i['qty']}건 ×{i['unit_price']:,}원" for i in c["items"]
    )


calc_query = (
    "다음 출장비를 정산하세요. 4명 소계, 소계의 10% 부가세, 부가세 포함 총액, 1인 평균(소계÷인원, 반올림)을 calc 도구로 계산하세요.\n"
    + "\n".join(f"- {c['name']}: {_fmt(c)}" for c in claims["claims"])
    + '\n최종 답은 JSON으로만: {"subtotal": 정수, "vat": 정수, "total_with_vat": 정수, "average": 정수}'
)
print("정답(파이썬 검증):", GROUND_TRUTH)

# (en) Default 1 trial for CI/automated runs; set ORCHESTRATION_CALC_TRIALS=3 in .env for statistical comparison.
# (kr) CI·자동 실행 기본 1 trial; 통계 비교는 .env 에 ORCHESTRATION_CALC_TRIALS=3.
N_TRIALS = max(1, int(os.environ.get("ORCHESTRATION_CALC_TRIALS", "1")))
single_ok = multi_ok = 0
calc_trials = []
for i in range(N_TRIALS):
    s = run_single(calc_query, reg_calc, extract_calc)
    m = run_multi(calc_query, reg_calc, extract_calc, verify_calc)
    s_correct = not verify_calc(s["answer"])
    m_correct = not verify_calc(m["answer"])
    single_ok += int(s_correct)
    multi_ok += int(m_correct)
    s_total = (
        s["answer"].get("total_with_vat") if isinstance(s["answer"], dict) else None
    )
    m_total = (
        m["answer"].get("total_with_vat") if isinstance(m["answer"], dict) else None
    )
    calc_trials.append(
        {
            "single_total": s_total,
            "single_correct": s_correct,
            "multi_total": m_total,
            "multi_correct": m_correct,
            "verifier_caught": m["revised"],
        }
    )
    print(
        f"[trial {i}] single total={s_total} correct={s_correct} | multi total={m_total} correct={m_correct} (검증 보완={m['revised']})"
    )
print(f"=> 정답률: single {single_ok}/{N_TRIALS} · multi {multi_ok}/{N_TRIALS}")

# (en) The verifier is deterministic: here it flags a deliberately wrong total a single agent would ship unnoticed.
# (kr) 검증기는 결정적으로 동작합니다: 아래는 일부러 틀린 총액을 검증기가 잡아내는 예입니다(단일 에이전트라면 모른 채 내보냈을 값).
planted = {
    "subtotal": GT_SUBTOTAL,
    "vat": GT_VAT,
    "total_with_vat": GT_TOTAL - 50000,
    "average": GT_AVG,
}
print("검증기 데모 — 틀린 총액", planted["total_with_vat"], "입력 (정답", GT_TOTAL, ")")
print("  → 검증기가 잡은 문제:", verify_calc(planted))

**출력 해석:** 검증 가능한 계산에서 멀티 롤의 값은 **모든 출력이 검증을 거친다**는 점입니다.

- `정답률: single N/M · multi N/M` (M = `ORCHESTRATION_CALC_TRIALS`, 기본 1) — 이번 실행에서는 single도 다 맞았을 수 있습니다. EXAONE는 `calc` 도구가 있으면 이 정도 정산은 대체로 잘합니다. **하지만 single은 검증 없이 내보냅니다** — `calc`가 있어도 다단계 산술은 가끔 틀리는데, 그러면 틀린 줄 모르고 내보냅니다.
- `검증기 데모 — 틀린 총액 … → 검증기가 잡은 문제: [...]` — 결정적 검증기(파이썬 재계산)는 이런 불일치를 **빠짐없이** 잡아냅니다. multi는 이 검증을 매 출력에 적용하고, 틀리면 `calc`로 **보완을 시도한 뒤 다시 검증**(`final_ok`)합니다.
- 즉 multi의 이점은 "더 정확한 계산"이 아니라 **모든 답이 검증을 거친다**는 점입니다 — single은 틀려도 모른 채 내보내지만 multi는 적어도 압니다. 대가는 지연(검증 + 보완). **출력이 검증 가능하고 틀리면 비용이 큰 작업(정산·집계·규정)**에서 그 가치가 큽니다.

### Session 3-4. 산출물 — `comparison.json` + 결정 가이드

**하는 일:** Part A·B의 결과를 `comparison.json`에 저장하고, "언제 single을 쓰고 언제 multi를 쓸지" 결정 가이드 `decision_tree.md`를 만듭니다.

**의미:** 두 파트의 숫자(지연·품질·정답률)가 결정 기준의 **근거**가 됩니다 — 추측이 아니라 측정 결과로 방식을 고릅니다.


In [ ]:
comparison = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "has_api": HAS_API,
    "part_a_well_specified": results_a,
    "part_b_calculation": {
        "n_trials": N_TRIALS,
        "single_correct": single_ok,
        "multi_correct": multi_ok,
        "ground_truth_total": GT_TOTAL,
        "trials": calc_trials,
    },
}
(out_dir / "comparison.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8"
)
(out_dir / "decision_tree.md").write_text(
    (DATA / "decision_tree_template.md").read_text(encoding="utf-8"), encoding="utf-8"
)
print("saved:", (out_dir / "comparison.json").resolve())
print("saved:", (out_dir / "decision_tree.md").resolve())

**출력 해석:** 비교 결과와 결정 가이드가 저장되었습니다.

- `comparison.json` — Part A(지연·품질)와 Part B(정답률)가 한 파일에 정리됩니다. 단일/멀티 선택의 데이터 근거입니다.
- `decision_tree.md` — **기본은 single**(빠르고 대개 충분), **multi는** 출력이 검증 가능하고 틀리면 비용이 큰 작업(계산·정산·규정 준수)이나 단계별 감사·역할 분리가 필요할 때 씁니다. 템플릿에서 생성되어 팀 wiki에 바로 붙일 수 있습니다.


## 체크포인트

- [ ] Session 1 `workflow_trace.json` — planner/executor/critic 세 단계의 success·latency와 executor `stop_reason`이 기록된다(프롬프트에 JSON 형태를 명시하면 planner·critic도 거의 항상 success).
- [ ] Session 2 `routing_table.json` — Stage 1(규칙) vs Stage 2(규칙+LLM 분류기) 일치율이 행 단위로 기록되고, 모호 질의에서 LLM 분류기가 일치율을 끌어올린다.
- [ ] Session 3 `comparison.json` — Part A(잘 명세된 작업)는 single·multi 품질이 같고 single이 더 빠르며, Part B(계산)는 멀티 롤이 결정적 검증기로 오류를 잡아낸다(single은 검증 없이 내보낸다).
- [ ] Session 3 `decision_tree.md` — 템플릿에서 생성된다(이 노트북은 첫 셀에서 LLM 키가 필수).


## Wrap-up. 마무리

이 노트북에서는 사내 정책 메모라는 한 과제를 **세 가지 조율 관점**으로 다뤘습니다.

- **Session 1 — 역할 분리:** Planner(계획)→Executor(도구로 근거 수집·초안)→Critic(체크리스트 검수)를 명시적으로 돌리고 단계별 trace를 남겼습니다. 구조화 출력은 단계 간 계약이지만, `StructuredOutputPipeline`은 사후 검증기이므로 **프롬프트에 목표 JSON 형태를 명시**해야 planner·critic이 안정적으로 성공한다는 점을 확인했습니다.
- **Session 2 — 역할로 라우팅:** 5개 프로필 라우팅을 **두 가지 타입**으로 봤습니다 — 결정적 규칙은 싸고 키워드 질의에 강하며, **LLM 5-way 분류기**는 뜻으로 분류해 모호한 질의까지 다룹니다. 경쟁이 아니라 입력 성격에 맞춰 결합하는 **보완** 관계입니다.
- **Session 3 — 언제 무엇을 쓸까:** 같은 작업을 단일 방식과 멀티 롤(executor+검증+보완)로 비교했습니다. 잘 명세된 정책 작업(Part A)은 single이 빠르고 품질도 같아 멀티 롤이 순손실이지만, **검증 가능한 계산(Part B)에서는 멀티 롤이 결정적 검증기로 모든 출력을 확인합니다 — single은 검증 없이 내보내지만 multi는 산술 오류를 잡아 보완을 시도하고 다시 검증합니다.**

**핵심 takeaways**
- 멀티 에이전트는 목적이 아니라 **도구**입니다 — 계획·초안·독립 검수가 필요하거나 단계별 권한·trace가 필요할 때만 역할을 나눕니다.
- 구조화 출력은 스키마를 **프롬프트에도** 명시해야 동작합니다(검증기만으로는 부족).
- 라우팅은 **싸고 결정적인 규칙을 먼저, 못 정하는 모호한 입력에서만 LLM 분류기**를 — 비용과 정확도를 함께 잡습니다.
- 멀티 롤의 값은 '더 뛰어난 생성'이 아니라 **독립 검증**입니다 — 검증기가 결정적일수록(계산·스키마) 신뢰도가 높고, 능력 격차(산술)는 멀티 롤이 아니라 **도구**로 메웁니다.

**다음:** **Track 07 — Safety, HITL & Observability**(여기서 남긴 workflow trace가 관측으로 이어집니다). 같은 조율 문제를 다른 하니스로 보려면 **Track 09(LangGraph)**, 의사결정 트리로 캡스톤을 고르려면 **Track 10**으로 이어집니다.
